- Verificar se a anotação tem dimensões plausiveis (mesma resolução e pontos dentro da imagem)
- Verificar se as mascaras estão invertidas
- Métricas entre anotadores:
     - máscaras: Considerar uma máscara como real e usar IoU ou Dice para calcular a discrepância (fazer por vértebra?)
     - pontos: distância entre os pontos (considerar separado C2 e C4) 
- Métricas por par de anotadores (matriz de correlação mais ou menos, só que com as métricas)
- Salvar cada comparação de cada frame (uma pasta para máscara, uma pasta para C2 e uma pasta para C4)

### **Comparando Anotações de um Mesmo Frame**

In [ ]:
import sys
sys.path.append('../')

In [ ]:
import pandas as pd
import os
import numpy as np
import random
import torch
import math
import matplotlib.pyplot as plt
from scipy import stats
import cv2

from src.data_extraction.labels import get_label_metadata
from src.data_extraction.attributions import get_attributions_metadata
from src.data_extraction.video_frame import create_video_frame_metadata_from_label_and_attributions
from src.create_folders import get_frame_from_video
from src.target.points import load_points

In [ ]:
def set_seed(seed_value=42):
    random.seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)

set_seed(42)

### **Importando Dados**

In [ ]:
labels_dir = '..\\data\\rotulos\\anotacoes-tecgraf\\'
frames_dir = '..\\data\\frames\\'
videos_dir = '..\\data\\videos\\' 

In [ ]:
# Importando dados:
attribution_metadata_df = get_attributions_metadata(labels_dir)
labels_metadata_df = get_label_metadata(labels_dir)

video_frame_metadata_df = create_video_frame_metadata_from_label_and_attributions(
    label_metadata=labels_metadata_df,
    attributions_metadata=attribution_metadata_df
)
video_frame_metadata_df

### **Verifica dimensões**

In [ ]:
# Verificando resoluções das Máscaras
def check_mask_resolution(video_frame, label_folder, video_folder, video_id, frame_id):
    path_label = os.path.join(label_folder, "Mask.tif")
    path_video = os.path.join(video_folder, f"{video_id}.avi")
    
    frame = get_frame_from_video(path_video, int(frame_id))
    mask = cv2.imread(path_label)
    
    if frame is None:
        raise Exception("Frame não encontrado\n\t"
                        f"Frame: {video_frame}\n\t" \
                        f"Video Path: {path_video}")
    if mask is None:
        raise Exception("Máscara não encontrada\n\t"
                        f"Frame: {video_frame}\n\t" \
                        f"Label Path: {path_label}")
    if frame.shape != mask.shape:
        print(f"\nResolução de Máscara Incorreta\n\t" \
              f"Frame: {video_frame}\n\t" \
              f"Frame Original: {frame.shape}\n\t" \
              f"Máscara Rotulada: {mask.shape}\n\t" \
              f"Label Folder: {label_folder}")

    return

# Verificando Posições dos Pontos
def check_point_position(video_frame, label_folder, video_folder, video_id, frame_id):
    path_video = os.path.join(video_folder, f"{video_id}.avi")

    frame = get_frame_from_video(path_video, int(frame_id))
    points = load_points(label_folder, "Results.csv")
    h, w, _ = frame.shape
    
    for point in points:
        if h <= point[0] or w <= point[1]:
            print(f"\nPosição de Ponto Inválida\n\t" \
                  f"Frame: {video_frame}\n\t" \
                  f"Frame Original: {frame.shape}\n\t" \
                  f"Pontos Marcados: {point}\n\t" \
                  f"Label Folder: {label_folder}")

    return

# Aplicando verificações iniciais
def verifica_dimensoes(row, video_folder):
    has_mask = row["has_mask"]
    has_points = row["has_points"]
    video_frame = row["video_frame"]
    video_id = row["video_id"]
    frame_id = row["frame_id"]
    file_path = row["file_path"]

    if has_mask:
        check_mask_resolution(video_frame, file_path, video_folder, video_id, frame_id)
    if has_points:
        check_point_position(video_frame, file_path, video_folder, video_id, frame_id)

video_frame_metadata_df.apply(verifica_dimensoes, axis =1, args = (videos_dir,))
print("Primeira Etapa Concluída")

### **Checa por Máscaras Invertidas**

In [ ]:
# Checa Quais Máscaras Estão Invertidas

def invert_color_mask(mask):
    mask_invert = cv2.bitwise_not(mask)
    return mask_invert

def check_inverted_masks(row):
    has_mask = row["has_mask"]
    video_frame = row["video_frame"]
    file_path = row["file_path"]

    if has_mask:
        path_label = os.path.join(file_path, "Mask.tif")
        mask = cv2.imread(path_label)
        moda = stats.mode(mask, axis=None)
        if moda.mode == 255:
            print(f"\nMáscara Invertida\n\t" \
                  f"Frame: {video_frame}\n\t" \
                  f"Label Folder: {file_path}\n\t" \
                  f"Moda: {moda.mode}")
            mask_invert = invert_color_mask(mask)
            fig, ax = plt.subplots(1,2)
            ax[0].imshow(mask, cmap='gray')
            ax[0].axis('off')
            ax[1].imshow(mask_invert, cmap = "gray")
            ax[1].axis('off')
            plt.show()
        elif moda.mode != 0:
            print(f"\nModa da máscara Incoerente (diferente de 0 ou 1)\n\t" \
                  f"Frame: {video_frame}\n\t" \
                  f"Label Folder: {file_path}\n\t" \
                  f"Moda: {moda.mode}")
            plt.imshow(mask, cmap='gray')
            plt.axis('off')
            plt.show()
    return

video_frame_metadata_df.apply(check_inverted_masks, axis =1)
print("Checagem concluída!")

In [ ]:
# Corrige e Salva Quais Máscaras Estão Invertidas

def correct_inverted_masks(row):
    has_mask = row["has_mask"]
    video_frame = row["video_frame"]
    file_path = row["file_path"]

    if has_mask:
        path_label = os.path.join(file_path, "Mask.tif")
        mask = cv2.imread(path_label)
        moda = stats.mode(mask, axis=None)
        if moda.mode == 255:
            mask_invert = invert_color_mask(mask)
            cv2.imwrite(path_label, mask_invert)
            print(f"\nMáscara Invertida\n\t" \
                  f"Frame: {video_frame}\n\t" \
                  f"Label Folder: {file_path}\n\t" \
                  f"Moda: {moda.mode}")
        elif moda.mode != 0:
            print(f"\nModa da máscara Incoerente (diferente de 0 ou 1)\n\t" \
                  f"Frame: {video_frame}\n\t" \
                  f"Label Folder: {file_path}\n\t" \
                  f"Moda: {moda.mode}")
            plt.imshow(mask, cmap='gray')
            plt.axis('off')
            plt.show()
    return

substituir_definitivamente = False
if substituir_definitivamente:
    video_frame_metadata_df.apply(correct_inverted_masks, axis =1)
    print("Correção de Máscaras Invertidas Concluída!")


### **Visualizando Anotações de um Mesmo Frame**

In [ ]:
# Visualizando Marcações de Pontos
def generate_color_list(n, cmap_name = "Wistia"):
    cmap = plt.get_cmap(cmap_name)
    colors = [cmap(i) for i in np.linspace(0, 1, n)]
    return colors 

def check_coherence_points(df_video_frame, video_folder, output_folder):
    n_rotuladores = df_video_frame.shape[0]
    colors = generate_color_list(n_rotuladores)

    video_id = df_video_frame["video_id"].values[0]
    frame_id = df_video_frame["frame_id"].values[0]
    video_frame = df_video_frame["video_frame"].values[0]

    path_video = os.path.join(video_folder, f"{video_id}.avi")
    frame = get_frame_from_video(path_video, int(frame_id))
    
    fig, ax = plt.subplots(1,n_rotuladores+1, figsize=(5*(n_rotuladores+1),5))
    
    # Plot por Rotulador
    all_points = []
    all_labelers = []
    all_distance = []
    for i in range(n_rotuladores):
        df_rotulador = df_video_frame.loc[i]
        labeler = df_rotulador['labeler']
        ax[i].imshow(frame)
        if df_rotulador["has_points"]:
            points = load_points(df_rotulador["file_path"], "Results.csv")
            distance = math.sqrt((points[1][0] - points[0][0])**2 + (points[1][1] - points[0][1])**2)
            ponto_medio = (points[0] + points[1]) / 2

            ax[i].scatter(points[:, 0], points[:, 1], color=colors[i], marker='o', s=5, linewidths=1)
            ax[i].plot(points[:, 0], points[:, 1], color=colors[i], lw=1, alpha = 0.3)
            ax[i].text(ponto_medio[0] + 10, ponto_medio[1], f'{distance:.1f} px', 
                       color=colors[i], fontweight='bold', fontsize=12)
            all_points.append(points)
            all_labelers.append(labeler)
            all_distance.append(distance)
        else:
            ax[i].text(frame.shape[1]/2, frame.shape[0]/2, 'NÃO ROTULADO', 
                       color='red', fontweight='bold', fontsize=20, ha='center', va='center')
        ax[i].axis("off")
        ax[i].set_title(f"{labeler} - {video_frame}")

    # Plot Geral 
    ax[n_rotuladores].imshow(frame)
    for i in range(len(all_points)):
        points = all_points[i]
        ax[n_rotuladores].scatter(points[:, 0], points[:, 1], color=colors[i],
                                  marker='o', s=5, linewidths=1, label = f"{all_labelers[i]} - ({all_distance[i]:.1f} px)")
        ax[n_rotuladores].plot(points[:, 0], points[:, 1], color=colors[i], lw=1, alpha = 0.3)
    ax[n_rotuladores].axis("off")
    ax[n_rotuladores].set_title("Sobreposição de Pontos")
    ax[n_rotuladores].legend()
    
    plt.tight_layout()
    if output_folder:
        plt.savefig(os.path.join(output_folder, video_frame + ".png"))
    plt.show()
    
    return 

output_folder = r"..\\fig\\comparison\\point"
video_frame_unique = video_frame_metadata_df["video_frame"].unique()
for video_frame in video_frame_unique:
    df_video_frame = video_frame_metadata_df.loc[video_frame_metadata_df["video_frame"] == video_frame].reset_index()
    check_coherence_points(df_video_frame, videos_dir, output_folder)

In [ ]:
# Visualizando Marcações de Máscaras

def generate_color_list(n, cmap_name = "Wistia"):
    cmap = plt.get_cmap(cmap_name)
    colors = [cmap(i) for i in np.linspace(0, 1, n)]
    return colors 

def check_coherence_masks(df_video_frame, video_folder, output_dir = "", alpha=0.4):
    n_rotuladores = df_video_frame.shape[0]
    colors = generate_color_list(n_rotuladores)

    video_id = df_video_frame["video_id"].values[0]
    frame_id = df_video_frame["frame_id"].values[0]
    video_frame = df_video_frame["video_frame"].values[0]

    path_video = os.path.join(video_folder, f"{video_id}.avi")
    frame = get_frame_from_video(path_video, int(frame_id))
    
    fig, ax = plt.subplots(1, n_rotuladores + 1, figsize=(5 * (n_rotuladores + 1), 5))
    
    all_masks = []
    all_labelers = []

    for i in range(n_rotuladores):
        df_rotulador = df_video_frame.iloc[i]
        labeler = df_rotulador['labeler']
        
        ax[i].imshow(frame, cmap='gray')
        
        if df_rotulador["has_mask"]:

            path_label = os.path.join(df_rotulador["file_path"], "Mask.tif")
            mask = cv2.imread(path_label, cv2.IMREAD_UNCHANGED)
            if mask.ndim == 3:
                mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

            if mask.shape[:2] != frame.shape[:2]:
                ax[i].text(frame.shape[1]/2, frame.shape[0]/2, 'FORMATO\nINCORRETO',
                        color='red', fontweight='bold', fontsize=20, ha='center', va='center')
            else:
                # Criando a sobreposição colorida
                mask_rgba = np.zeros((mask.shape[0], mask.shape[1], 4))
                rgb = plt.cm.colors.to_rgb(colors[i])
                mask_rgba[..., :3] = rgb
                mask_rgba[..., 3] = (mask > 0) * alpha

                ax[i].imshow(mask_rgba)
                all_masks.append(mask)
                all_labelers.append(labeler)
        else:
            ax[i].text(frame.shape[1]/2, frame.shape[0]/2, 'SEM MÁSCARA', 
                       color='red', fontweight='bold', fontsize=20, ha='center', va='center')
        
        ax[i].axis("off")
        ax[i].set_title(f"{labeler} - {video_frame}")

    # Plot Geral
    ax[n_rotuladores].imshow(frame, cmap='gray')
    for i, mask in enumerate(all_masks):
        mask_rgba = np.zeros((*mask.shape, 4))
        rgb = plt.cm.colors.to_rgb(colors[i])
        mask_rgba[..., :3] = rgb
        mask_rgba[..., 3] = (mask > 0) * alpha
        ax[n_rotuladores].imshow(mask_rgba, label=all_labelers[i])
        
    ax[n_rotuladores].axis("off")
    ax[n_rotuladores].set_title("Sobreposição de Máscaras")
    
    if output_dir:
        plt.savefig(os.path.join(output_dir,video_frame + ".png"))
    plt.show()

output_dir = r"..\\fig\\comparison\\mask"
video_frame_unique = video_frame_metadata_df["video_frame"].unique()
for video_frame in video_frame_unique[100:200]:
    df_video_frame = video_frame_metadata_df.loc[video_frame_metadata_df["video_frame"] == video_frame].reset_index()
    check_coherence_masks(df_video_frame, videos_dir, output_dir)